# Lekcja 11 - Protokół agent-do-agenta (A2A)


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Czym jest protokół A2A?

The **Protokół Agent-do-Agenta (A2A)** jest otwartym standardem, który umożliwia agentom AI komunikację,
odkrywanie się nawzajem i współpracę — nawet gdy są zbudowani na różnych frameworkach lub hostowani
przez różne usługi.

Key concepts:

- **Discovery** – Agenci publikują *Kartę Agenta*, która opisuje ich możliwości, ułatwiając innym agentom (lub orkiestratorom) znalezienie odpowiedniego specjalisty do zadania.
- **Message Passing** – Agenci wymieniają się ustrukturyzowanymi wiadomościami za pośrednictwem wspólnego protokołu, dzięki czemu żądanie od jednego agenta może być zrozumiane i zrealizowane przez innego niezależnie od jego wewnętrznej implementacji.
- **Task Lifecycle** – A2A definiuje stany takie jak *submitted*, *working*, *completed*, i *failed*, dając orkiestratorowi pełną widoczność tego, jak postępuje przekazane zadanie.

W tej lekcji symulujemy współpracę w stylu A2A, łącząc trzy wyspecjalizowane agentów podróżniczych
w przepływ pracy, w którym każdy agent wnosi swoją wiedzę i przekazuje wyniki następnemu.


## Tworzenie wyspecjalizowanych agentów podróży


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Współpraca wieloagentowa za pomocą przepływu pracy

Łączymy trzy agenty w sekwencyjny przepływ pracy, który odzwierciedla przesyłanie wiadomości A2A:

1. **CurrencyExchangeAgent** otrzymuje żądanie użytkownika i przygotowuje wskazówki dotyczące waluty.
2. **ActivityPlannerAgent** otrzymuje wzbogacony kontekst i dodaje rekomendacje aktywności.
3. **TravelManagerAgent** łączy oba wejścia w ostateczne podsumowanie podróży.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Zrozumienie A2A w środowisku produkcyjnym

W środowisku produkcyjnym protokół A2A udostępnia potężne międzyserwisowe scenariusze:

| Możliwość | Opis |
|---|---|
| **Interoperacyjność między frameworkami** | Agent zbudowany w jednym frameworku może delegować zadania agentowi zbudowanemu w dowolnym innym frameworku zgodnym z A2A, umożliwiając prawdziwą interoperacyjność między organizacjami. |
| **Granice usług** | Agenci mogą działać w oddzielnych mikroserwisach, regionach chmurowych, a nawet w różnych organizacjach, nadal współpracując bezproblemowo. |
| **Dynamiczne odkrywanie** | Orkiestrator może w czasie działania zapytać rejestr Agent Card, aby znaleźć najlepiej dopasowanego specjalistę do danego podzadania. |
| **Strumieniowanie & powiadomienia push** | A2A obsługuje Server-Sent Events (SSE) dla aktualizacji postępu w czasie rzeczywistym oraz powiadomienia push dla zadań długotrwałych. |

Przepływ pracy, który zbudowaliśmy powyżej, jest uproszczoną, działającą w tym samym procesie wersją tego wzorca. W rzeczywistym
wdrożeniu każdy agent wystawiałby punkt końcowy HTTP, publikował Agent Card i komunikował się
za pomocą protokołu A2A JSON-RPC.


## Podsumowanie

W tej lekcji dowiedziałeś się:

1. **Co to jest protokół A2A** — otwarty standard do odkrywania agentów, przesyłania wiadomości oraz zarządzania zadaniami.
2. **Jak tworzyć wyspecjalizowane agenty** — agenta Currency Exchange, agenta Activity Planner oraz orkiestratora Travel Manager.
3. **Jak łączyć agenty w przepływ pracy** — używając `WorkflowBuilder` do modelowania sekwencyjnego przesyłania wiadomości między agentami.
4. **Jak A2A działa w produkcji** — umożliwiając współpracę między różnymi frameworkami i usługami dzięki dynamicznemu odkrywaniu i strumieniowym aktualizacjom.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Ten dokument został przetłumaczony przy użyciu usługi tłumaczeń AI [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy starań o poprawność, prosimy pamiętać, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Dokument oryginalny w języku macierzystym powinien być uznawany za źródło nadrzędne. W przypadku informacji o istotnym znaczeniu zaleca się skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
